## C0 - Loading and Evaluating Your Model
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 22, 2026

### About: In B4 you trained a Free/Blocked classifier. Today you'll load it back up and put it to the test -- first against images from your dataset, then live while you drive your robot around, right up to real obstacles.
### This is the first step toward autonomy: before your robot can act on what it sees, you need to know how well it actually sees.

### In B4 you trained one or more Free/Blocked models and saved them to your Models folder, along with a plot and a log entry for each one. Now let's load one back up and see how well it actually works.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

import ipywidgets as widgets
from IPython.display import display
from ipyfilechooser import FileChooser

from jetcam_lite import TraitletCamera, bgr8_to_jpeg

from robot_utils import get_rvr, close_if_exists
from gamepad_utils import connect_gamepad, start_control_loop, stop_control_loop
from jupyter_utils import register_dlink
from inference_utils import (
    load_model_and_metadata,
    show_inference_grid,
    start_live_classification,
    stop_live_classification,
)

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY C0.1: Load and Test Your Model</span>

### Choose a model to load below. Use the file browser to navigate to your `Models` folder (it should open there by default) and select a `.pth` file.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

model_chooser = FileChooser('/home/explorer/Models/')
model_chooser.filter_pattern = '*.pth'
display(model_chooser)

### Once you've selected a model above, run the next cell to load it. You should see a short summary of how that model was trained.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

model, device, class_names, training_record = load_model_and_metadata(model_chooser.selected)

## Testing against your dataset
### Let's run your model against 8 random images from your dataset and see how it does. Each image shows the *true* label (what it actually is) and the model's *predicted* label -- green means it got it right, red means it got it wrong.
### Note: since we didn't set aside a separate, never-seen test set for this activity, some of these images may be ones the model already trained on -- so this isn't a perfect measure of how well it would do on a brand new image. Still a great way to spot patterns in what it gets right and wrong!
### Run the cell below as many times as you like -- each time picks a new random sample.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
##### Re-run this cell any time for a new random sample! #####

show_inference_grid(model, class_names, device, training_record['dataset_dir'])

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY C0.2: Live Test -- Drive and Watch It Think</span>

## Live inference while you drive
### This time, you'll teleoperate your robot with the gamepad while your model classifies the path ahead in real time. Drive around, then drive slowly toward obstacles and watch the indicator below the camera feed change as the path goes from clear to blocked.
### This is still just *watching* the model think -- your robot isn't reacting to its own predictions yet. That's exactly what C1 builds next.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# Set up the robot connection, camera feed, and a big status indicator
# showing the model's current live prediction.

rvr = get_rvr()

camera = TraitletCamera()
camera.start()

image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

# A big rectangular status indicator -- green "Path Free" or red "Path Blocked",
# updated live as the model classifies each new frame.
status_indicator = widgets.Button(
    description='Path Free',
    button_style='success',
    disabled=True,
    layout=widgets.Layout(width='95%', height='80px')
)

display(widgets.VBox([image_widget, status_indicator]))

### Now connect the gamepad so you can drive, and start live classification so the indicator above starts updating.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

gamepad, gamepad_state = connect_gamepad()

left_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Left Stick:',
                                         orientation='vertical', readout=True, readout_format='.2f')
right_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Right Stick:',
                                          orientation='vertical', readout=True, readout_format='.2f')
display(widgets.HBox([left_stick_slider, right_stick_slider]))


def update_status_indicator(label, confidence):
    if label == 'free':
        status_indicator.description = 'Path Free'
        status_indicator.button_style = 'success'
    else:
        status_indicator.description = 'Path Blocked'
        status_indicator.button_style = 'danger'


def update_sticks(left_value, right_value):
    left_stick_slider.value = left_value
    right_stick_slider.value = right_value


start_live_classification(camera, model, class_names, device, update_status_indicator)
start_control_loop(rvr, gamepad_state, on_update=update_sticks)

### Drive around freely, then drive slowly toward a few different obstacles and watch the indicator. Try some tricky cases too -- an obstacle at an angle, low light, something near the edge of the frame.
### When you're done, run the cell below to stop driving and stop live classification.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

stop_control_loop()
stop_live_classification(camera)

<span style="color: green; font-size: 55px; font-style: italic;">Student Discussion Time</span>

### Talk through these questions with your team:
### - Were there moments the model was confidently wrong? What might have confused it -- lighting, obstacle angle, distance, background, something else?
### - Was the model more accurate on the dataset grid or live while driving? Why might those differ?
### - Look back at the training summary printed above (`training_record`) -- how many images did this model train on, and for how many epochs? Do you think more images, more epochs, or a different learning rate would have helped most?
### - If you trained more than one model in B4, load a different one above and repeat this notebook. Which one performs better, and why do you think that is?

## Great work! You now have direct evidence of how well your Free/Blocked model actually performs.

## **NEXT**: In C1, your robot will start acting on this model's predictions instead of just displaying them -- its first autonomous behavior.

In [ ]:
#### ------> RUN THIS CELL WHEN YOU'RE DONE WITH THIS NOTEBOOK <-----#####
close_if_exists()
print("Robot connection closed. Safe to move on to the next notebook!")